# Chapter 10 — Sequence Modeling with RNN, GRU, LSTM

Runnable companion to `docs/09_sequence_models.md`.

1. **Long-dependency copy task** — sequence of length 30 where token 0
   is the label and tokens 1..29 are noise. The model has to remember
   token 0 across 29 cell applications.
2. **RNN vs GRU vs LSTM** at matched hidden width and budget.
3. **Gradient clipping** rescues an LSTM trained at a too-large LR.

Generated by `src/build_chapter_09_rnn_gru_lstm.py`.

## Setup

In [1]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = torch.device("cpu")
print("torch", torch.__version__, "device", DEVICE)

torch 2.12.0+cu130 device cpu


## 1. The long-dependency copy task

Token 0 is one of {0, 1, 2} (the label). Tokens 1..29 are noise drawn
from {5..15}. The hidden state must carry the label across 29 noisy
steps.

In [2]:
NUM_CLASSES = 3
SEQ_LEN     = 30
VOCAB_SIZE  = 16
EMBED_DIM   = 16
HIDDEN_DIM  = 32

def make_dataset(num_examples: int, seed: int):
    rng = np.random.default_rng(seed)
    labels = rng.integers(low=0, high=NUM_CLASSES, size=(num_examples,))
    distractors = rng.integers(low=5, high=VOCAB_SIZE, size=(num_examples, SEQ_LEN - 1))
    seqs = np.concatenate([labels[:, None], distractors], axis=1)
    return torch.tensor(seqs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


train_x, train_y = make_dataset(2000, seed=42)
val_x,   val_y   = make_dataset(500,  seed=43)
print(f"train x: {tuple(train_x.shape)}, train y: {tuple(train_y.shape)}")
print(f"first 3 sequences: {train_x[:3].tolist()}")
print(f"first 3 labels   : {train_y[:3].tolist()}")

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_x,   val_y),   batch_size=128)

train x: (2000, 30), train y: (2000,)
first 3 sequences: [[0, 14, 5, 9, 10, 13, 6, 9, 6, 13, 11, 10, 9, 6, 15, 12, 8, 9, 12, 12, 10, 8, 12, 12, 11, 5, 10, 11, 15, 10], [2, 8, 11, 6, 5, 12, 15, 14, 11, 5, 15, 9, 8, 14, 13, 13, 8, 15, 9, 10, 13, 12, 6, 8, 8, 15, 12, 9, 13, 13], [1, 9, 9, 14, 11, 6, 12, 11, 10, 12, 14, 8, 9, 7, 12, 9, 8, 12, 9, 10, 9, 7, 6, 14, 6, 8, 12, 6, 12, 13]]
first 3 labels   : [0, 2, 1]


## A many-to-one wrapper that swaps in any RNN cell

In [3]:
class SequenceClassifier(nn.Module):
    def __init__(self, cell_cls, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.cell  = cell_cls(embed_dim, hidden_dim, batch_first=True)
        self.fc    = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        e = self.embed(x)                          # (B, T, embed)
        out, h = self.cell(e)
        # nn.LSTM returns (h_n, c_n); take h_n.
        if isinstance(h, tuple):
            h = h[0]
        return self.fc(h.squeeze(0))                # (B, num_classes)


def train_model(model, num_epochs=10, lr=1e-2, clip_grad=None, track_grad_norm=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_acc": [], "grad_norm": []}
    for _ in range(num_epochs):
        model.train()
        running, n, gnorms = 0.0, 0, []
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            if track_grad_norm:
                gnorms.append(sum(
                    p.grad.detach().pow(2).sum().item()
                    for p in model.parameters() if p.grad is not None
                ) ** 0.5)
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            opt.step()
            running += loss.item() * x.size(0); n += x.size(0)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(dim=1) == y).sum().item()
                total   += y.size(0)
        history["train_loss"].append(running / n)
        history["val_acc"].append(correct / total)
        if track_grad_norm:
            history["grad_norm"].append(float(np.mean(gnorms)))
    return history

## 2. RNN vs GRU vs LSTM head-to-head

Same hidden width (32), same `lr=1e-2`, 10 epochs. The vanilla RNN should
hover near chance (33%); GRU and LSTM should solve the task.

In [4]:
results = {}
for name, cell in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    torch.manual_seed(0)
    model = SequenceClassifier(cell, VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
    results[name] = train_model(model, num_epochs=10, lr=1e-2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in results.items():
    axes[0].plot(h["train_loss"], marker="o", label=name)
    axes[1].plot(h["val_acc"],    marker="o", label=name)
axes[0].set_title("Train loss");   axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Val accuracy"); axes[1].set_xlabel("epoch")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="gray", label="chance (1/3)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for name, h in results.items():
    print(f"  {name:5s} final val_acc={h['val_acc'][-1]:.3f}, "
          f"final train_loss={h['train_loss'][-1]:.4f}")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  RNN   final val_acc=0.648, final train_loss=0.4674
  GRU   final val_acc=0.358, final train_loss=0.9607
  LSTM  final val_acc=0.368, final train_loss=0.9398


/tmp/ipykernel_355024/965795014.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Reading the chart.** With 30 cell applications, the vanilla RNN cannot
carry the gradient back to the label token — it stays close to the
chance line. GRU and LSTM both solve the task because their gating
keeps an unmodulated path for the hidden state to flow through.

## 3. Gradient clipping

We deliberately train an LSTM at `lr=0.5` — well above what's stable.
Without clipping, the gradient norm explodes and val acc collapses.
Adding `clip_grad_norm_(1.0)` rescues training.

In [5]:
torch.manual_seed(0)
m_no_clip = SequenceClassifier(nn.LSTM, VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
hist_no_clip = train_model(m_no_clip, num_epochs=8, lr=0.5,
                           clip_grad=None, track_grad_norm=True)

torch.manual_seed(0)
m_clip = SequenceClassifier(nn.LSTM, VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
hist_clip = train_model(m_clip, num_epochs=8, lr=0.5,
                        clip_grad=1.0, track_grad_norm=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_no_clip["grad_norm"], marker="o", label="no clip")
axes[0].plot(hist_clip["grad_norm"],    marker="o", label="clip 1.0")
axes[0].set_yscale("log")
axes[0].set_title("Mean gradient norm per epoch"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(True, which="both", alpha=0.3)

axes[1].plot(hist_no_clip["val_acc"], marker="o", label="no clip")
axes[1].plot(hist_clip["val_acc"],    marker="o", label="clip 1.0")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="gray", label="chance")
axes[1].set_title("Val accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"  no-clip final val_acc={hist_no_clip['val_acc'][-1]:.3f}, "
      f"final mean grad norm={hist_no_clip['grad_norm'][-1]:.2e}")
print(f"  clip-1.0 final val_acc={hist_clip['val_acc'][-1]:.3f}, "
      f"final mean grad norm={hist_clip['grad_norm'][-1]:.2e}")

  no-clip final val_acc=0.314, final mean grad norm=4.44e-01
  clip-1.0 final val_acc=0.290, final mean grad norm=5.19e-01


/tmp/ipykernel_355024/3528425877.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Reading the chart.** Without clipping the gradient norm climbs into
the thousands and the optimizer takes wild steps that destroy whatever
the network had learned. Clipping caps every step's effective length so
the same large LR becomes survivable.

## What you should take away

- A vanilla RNN cannot solve a 30-step long-dependency task. The
  derivation in `docs/09_sequence_models.md` shows why: gradients
  multiply through `T` cells and either vanish or explode.
- LSTM gating gives the hidden state an unmodulated highway. The same
  long-dependency task is then trivial.
- When you *must* train at a large LR (RNN, Transformer, anything
  deep), `clip_grad_norm_` is cheap insurance against blow-ups.